In [3]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM


from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    GlobalMaxPooling1D
)

from tensorflow.keras.optimizers import Adam
from scipy.stats import t


In [4]:
DATASET_PATH = r"/kaggle/input/datasets/abubakarsiddiquemahi/url-phish-111k-dataset/Dataset.csv"

BATCH_SIZE = 16

LEARNING_RATE = 0.001

EPOCHS = 30

SEEDS = [42,3,7,72,82]


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    tf.random.set_seed(seed)


# ------------------------------------------------------------
# LOAD DATASET
# ------------------------------------------------------------

data = pd.read_csv(DATASET_PATH)

print(data.shape)

print(data.head())

print(data.columns)


# ------------------------------------------------------------
# HANDLE MISSING VALUES
# ------------------------------------------------------------

numeric_cols = data.select_dtypes(include=['number']).columns

categorical_cols = data.select_dtypes(exclude=['number']).columns


data[numeric_cols] = data[numeric_cols].fillna(
    data[numeric_cols].mean()
)


data[categorical_cols] = data[categorical_cols].fillna(
    "unknown"
)


# ------------------------------------------------------------
# LABEL ENCODING
# ------------------------------------------------------------

encoder = LabelEncoder()

for col in categorical_cols:

    data[col] = encoder.fit_transform(data[col])


# ------------------------------------------------------------
# FEATURES AND LABEL
# ------------------------------------------------------------

X = data.drop("label", axis=1)

y = data["label"]


# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(X)


# ------------------------------------------------------------
# TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)


# ------------------------------------------------------------
# RESHAPE FOR 1D-CNN
# ------------------------------------------------------------

X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)


X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)


# ------------------------------------------------------------
# DISPLAY SHAPE
# ------------------------------------------------------------

print()

print("Training samples :", len(X_train))

print("Testing samples  :", len(X_test))

print()

print("Training shape :", X_train.shape)

print("Testing shape  :", X_test.shape)

(116600, 26)
                          url  url_len             dom  dom_len  is_ip  \
0    https://www.rmit.edu.au/       24     rmit.edu.au       11      0   
1  http://www.latrobe.edu.au/       26  latrobe.edu.au       14      0   
2     https://www.cqu.edu.au/       23      cqu.edu.au       10      0   
3         http://bond.edu.au/       19     bond.edu.au       11      0   
4      http://www.csu.edu.au/       22      csu.edu.au       10      0   

      tld  tld_len  subdom_cnt  letter_cnt  digit_cnt  ...  under_cnt  \
0  edu.au        6           1          17          0  ...          0   
1  edu.au        6           1          19          0  ...          0   
2  edu.au        6           1          16          0  ...          0   
3  edu.au        6           0          13          0  ...          0   
4  edu.au        6           1          15          0  ...          0   

   letter_ratio  digit_ratio  spec_ratio  is_https  slash_cnt   entropy  \
0      0.708333          0.0

# CNN

In [4]:



def create_model(input_shape):

    model = Sequential()

    model.add(
        Input(shape=input_shape)
    )

    model.add(
        Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Conv1D(
            filters=32,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        GlobalMaxPooling1D()
    )

    model.add(
        Dense(
            32,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

In [5]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (1D-CNN)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # Build 1D-CNN Model
    model = create_model(
        (X_train.shape[1], X_train.shape[2])
    )

    # Train Model
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # =====================================================
    # Test Loss
    # =====================================================

    evaluation = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    test_loss = evaluation[0]

    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1785901387.464120      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
  64/5830 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.7571 - auc: 0.4552 - loss: 0.5891 

I0000 00:00:1785901393.260569     141 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5830/5830 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.9004 - auc: 0.8674 - loss: 0.2723 - val_accuracy: 0.9408 - val_auc: 0.9613 - val_loss: 0.1582
Epoch 2/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9363 - auc: 0.9482 - loss: 0.1775 - val_accuracy: 0.9552 - val_auc: 0.9748 - val_loss: 0.1229
Epoch 3/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9465 - auc: 0.9619 - loss: 0.1513 - val_accuracy: 0.9619 - val_auc: 0.9799 - val_loss: 0.1066
Epoch 4/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9494 - auc: 0.9664 - loss: 0.1424 - val_accuracy: 0.9634 - val_auc: 0.9809 - val_loss: 0.1027
Epoch 5/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9521 - auc: 0.9697 - loss: 0.1349 - val_accuracy: 0.9655 - val_auc: 0.9831 - val_loss: 0.0968
Epoch 6/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9541 - auc: 0.9724 - loss: 0.1298 - val_accuracy: 0.9643 - val_auc: 0.9828 - val_loss: 0.0979
Epoch 7/30
5830/5830 ━━━━━━━━━━━━

In [6]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.074830  0.973756   0.947160  0.863855  0.903592  0.889701   
1     3  0.076412  0.972427   0.944832  0.856325  0.898404  0.883926   
2     7  0.077668  0.973113   0.931432  0.875602  0.902655  0.887645   
3    72  0.076299  0.972298   0.953836  0.846386  0.896904  0.883069   
4    82  0.077763  0.972727   0.958647  0.844880  0.898175  0.884849   

        AUC  Specificity  
0  0.990034      0.99200  
1  0.989688      0.99170  
2  0.989610      0.98930  
3  0.990350      0.99320  
4  0.989950      0.99395  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.076594  0.001199  0.0766 ± 0.0012  [0.0751, 0.0781]
1     Accuracy  0.972864  0.000589  0.9729 ± 0.0006  [0.9721, 0.9736]
2    Precision  0.947181  0.010360  0.9472 ± 0.0104  [0.9343, 0.9600]
3       Recall  0.857410  0.012769  0.8574 ± 0.0128  [0.8416, 0.8733]
4

# LSTM

In [19]:



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        LSTM(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        LSTM(
            64
        )
    )

    model.add(
        Dropout(0.5
        )
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [20]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (LSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build LSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 45s 7ms/step - accuracy: 0.9364 - auc: 0.9557 - loss: 0.1676 - val_accuracy: 0.9511 - val_auc: 0.9724 - val_loss: 0.1334
Epoch 2/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9652 - auc: 0.9812 - loss: 0.1023 - val_accuracy: 0.9709 - val_auc: 0.9868 - val_loss: 0.0831
Epoch 3/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9735 - auc: 0.9873 - loss: 0.0796 - val_accuracy: 0.9763 - val_auc: 0.9909 - val_loss: 0.0685
Epoch 4/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9768 - auc: 0.9902 - loss: 0.0691 - val_accuracy: 0.9779 - val_auc: 0.9897 - val_loss: 0.0667
Epoch 5/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9796 - auc: 0.9914 - loss: 0.0622 - val_accuracy: 0.9798 - val_auc: 0.9924 - val_loss: 0.0610
Epoch 6/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9805 - auc: 0.9926 - loss: 0.0571 - val_accuracy: 0.9794 - val_auc: 0.9926 - val_loss: 0.0600
Epoc

In [ ]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.059208  0.984520   0.961047  0.928916  0.944708  0.935894   
1     3  0.062031  0.983448   0.955025  0.927410  0.941015  0.931526   
2     7  0.059854  0.984777   0.958836  0.933133  0.945810  0.937073   
3    72  0.055754  0.984391   0.960436  0.928614  0.944257  0.935365   
4    82  0.053857  0.984434   0.952279  0.937651  0.944908  0.935883   

        AUC  Specificity  
0  0.994682      0.99375  
1  0.994015      0.99275  
2  0.995185      0.99335  
3  0.995145      0.99365  
4  0.994900      0.99220  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.058141  0.003288  0.0581 ± 0.0033  [0.0541, 0.0622]
1     Accuracy  0.984314  0.000507  0.9843 ± 0.0005  [0.9837, 0.9849]
2    Precision  0.957525  0.003754  0.9575 ± 0.0038  [0.9529, 0.9622]
3       Recall  0.931145  0.004231  0.9311 ± 0.0042  [0.9259, 0.9364]
4

# BiLSTM

In [5]:

from tensorflow.keras.layers import Bidirectional



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            LSTM(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            LSTM(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [6]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiLSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1785995200.166306      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 68s 11ms/step - accuracy: 0.9449 - auc: 0.9643 - loss: 0.1476 - val_accuracy: 0.9648 - val_auc: 0.9784 - val_loss: 0.1050
Epoch 2/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 62s 11ms/step - accuracy: 0.9721 - auc: 0.9862 - loss: 0.0835 - val_accuracy: 0.9755 - val_auc: 0.9889 - val_loss: 0.0741
Epoch 3/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 63s 11ms/step - accuracy: 0.9757 - auc: 0.9895 - loss: 0.0719 - val_accuracy: 0.9768 - val_auc: 0.9899 - val_loss: 0.0686
Epoch 4/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 63s 11ms/step - accuracy: 0.9787 - auc: 0.9914 - loss: 0.0641 - val_accuracy: 0.9789 - val_auc: 0.9910 - val_loss: 0.0638
Epoch 5/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 63s 11ms/step - accuracy: 0.9801 - auc: 0.9918 - loss: 0.0596 - val_accuracy: 0.9804 - val_auc: 0.9923 - val_loss: 0.0586
Epoch 6/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 63s 11ms/step - accuracy: 0.9813 - auc: 0.9930 - loss: 0.0556 - val_accuracy: 0.9813 - val_auc: 0.9929 - val_loss: 0.0564
Epoch 7/30
5830/

In [7]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",      
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.055475  0.984091   0.950504  0.937048  0.943728  0.934497   
1     3  0.057623  0.983491   0.955887  0.926807  0.941122  0.931675   
2     7  0.054766  0.984305   0.955302  0.933434  0.944241  0.935195   
3    72  0.059136  0.983148   0.946325  0.934639  0.940446  0.930655   
4    82  0.054211  0.984906   0.957178  0.935843  0.946390  0.937689   

        AUC  Specificity  
0  0.995714      0.99190  
1  0.995284      0.99290  
2  0.996122      0.99275  
3  0.994990      0.99120  
4  0.995230      0.99305  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.056242  0.002072  0.0562 ± 0.0021  [0.0537, 0.0588]
1     Accuracy  0.983988  0.000691  0.9840 ± 0.0007  [0.9831, 0.9848]
2    Precision  0.953039  0.004524  0.9530 ± 0.0045  [0.9474, 0.9587]
3       Recall  0.933554  0.004005  0.9336 ± 0.0040  [0.9286, 0.9385]
4

# GRU

In [3]:

from tensorflow.keras.layers import GRU



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        GRU(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        GRU(
            64
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [4]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (GRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1785929357.623551      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 48s 7ms/step - accuracy: 0.9412 - auc: 0.9604 - loss: 0.1570 - val_accuracy: 0.9616 - val_auc: 0.9809 - val_loss: 0.1078
Epoch 2/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 42s 7ms/step - accuracy: 0.9670 - auc: 0.9819 - loss: 0.0979 - val_accuracy: 0.9749 - val_auc: 0.9893 - val_loss: 0.0759
Epoch 3/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9739 - auc: 0.9874 - loss: 0.0789 - val_accuracy: 0.9760 - val_auc: 0.9907 - val_loss: 0.0708
Epoch 4/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 42s 7ms/step - accuracy: 0.9774 - auc: 0.9900 - loss: 0.0687 - val_accuracy: 0.9790 - val_auc: 0.9916 - val_loss: 0.0634
Epoch 5/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 42s 7ms/step - accuracy: 0.9792 - auc: 0.9910 - loss: 0.0628 - val_accuracy: 0.9790 - val_auc: 0.9933 - val_loss: 0.0628
Epoch 6/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - accuracy: 0.9813 - auc: 0.9926 - loss: 0.0576 - val_accuracy: 0.9808 - val_auc: 0.9919 - val_loss: 0.0601
Epoch 7/30
5830/5830 ━

In [5]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.051570  0.983533   0.952249  0.931024  0.941517  0.932016   
1     3  0.055089  0.984391   0.953931  0.935542  0.944647  0.935623   
2     7  0.055131  0.983105   0.956047  0.923795  0.939645  0.930011   
3    72  0.056315  0.983448   0.951941  0.930723  0.941212  0.931662   
4    82  0.051669  0.983362   0.954997  0.926807  0.940691  0.931159   

        AUC  Specificity  
0  0.995396      0.99225  
1  0.994976      0.99250  
2  0.994373      0.99295  
3  0.994382      0.99220  
4  0.995514      0.99275  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.053955  0.002188  0.0540 ± 0.0022  [0.0512, 0.0567]
1     Accuracy  0.983568  0.000487  0.9836 ± 0.0005  [0.9830, 0.9842]
2    Precision  0.953833  0.001758  0.9538 ± 0.0018  [0.9517, 0.9560]
3       Recall  0.929578  0.004475  0.9296 ± 0.0045  [0.9240, 0.9351]
4

# BiGRU

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf


def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            GRU(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            GRU(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [4]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiGRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1785937041.583809      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 67s 11ms/step - accuracy: 0.9536 - auc: 0.9727 - loss: 0.1279 - val_accuracy: 0.9730 - val_auc: 0.9869 - val_loss: 0.0783
Epoch 2/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 60s 10ms/step - accuracy: 0.9727 - auc: 0.9875 - loss: 0.0806 - val_accuracy: 0.9766 - val_auc: 0.9904 - val_loss: 0.0677
Epoch 3/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 60s 10ms/step - accuracy: 0.9756 - auc: 0.9897 - loss: 0.0718 - val_accuracy: 0.9788 - val_auc: 0.9919 - val_loss: 0.0609
Epoch 4/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 60s 10ms/step - accuracy: 0.9780 - auc: 0.9913 - loss: 0.0650 - val_accuracy: 0.9805 - val_auc: 0.9929 - val_loss: 0.0578
Epoch 5/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 61s 10ms/step - accuracy: 0.9792 - auc: 0.9919 - loss: 0.0617 - val_accuracy: 0.9821 - val_auc: 0.9914 - val_loss: 0.0580
Epoch 6/30
5830/5830 ━━━━━━━━━━━━━━━━━━━━ 61s 10ms/step - accuracy: 0.9808 - auc: 0.9924 - loss: 0.0579 - val_accuracy: 0.9813 - val_auc: 0.9930 - val_loss: 0.0558
Epoch 7/30
5830/

In [5]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.050330  0.984391   0.962164  0.926807  0.944155  0.935307   
1     3  0.052938  0.983491   0.961333  0.921084  0.940778  0.931479   
2     7  0.053966  0.982890   0.962623  0.915361  0.938397  0.928867   
3    72  0.050708  0.983748   0.957672  0.926807  0.941987  0.932709   
4    82  0.051099  0.984563   0.955946  0.934639  0.945172  0.936271   

        AUC  Specificity  
0  0.995782      0.99395  
1  0.995549      0.99385  
2  0.995620      0.99410  
3  0.995969      0.99320  
4  0.995457      0.99285  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.051808  0.001568  0.0518 ± 0.0016  [0.0499, 0.0538]
1     Accuracy  0.983816  0.000681  0.9838 ± 0.0007  [0.9830, 0.9847]
2    Precision  0.959947  0.002966  0.9599 ± 0.0030  [0.9563, 0.9636]
3       Recall  0.924940  0.007205  0.9249 ± 0.0072  [0.9160, 0.9339]
4